In [21]:
import polars as pl
import numpy as np
import pandas as pd

In [22]:
%cd C:/Users/citioplab/works/codes/baseball_projects/

C:\Users\citioplab\works\codes\baseball_projects


In [23]:
# Define a dictionary to group outcomes together
des_dict = {
    'ball': 'ball',
    'hit_into_play': 'hit_into_play',
    'called_strike': 'called_strike',
    'foul': 'foul',
    'swinging_strike': 'swinging_strike',
    'blocked_ball': 'ball',
    'swinging_strike_blocked': 'swinging_strike',
    'foul_tip': 'swinging_strike',
    'foul_bunt': 'foul',
    'hit_by_pitch': 'hit_by_pitch',
    'pitchout': 'ball',
    'missed_bunt': 'swinging_strike',
    'bunt_foul_tip': 'swinging_strike',
    'foul_pitchout': 'foul',
}

# Define a dictionary to group events together
ev_dict = {
    'single': 'single',
    'walk': 'walk',
    'strikeout': 'strikeout',
    'field_out': 'field_out',
    'force_out': 'field_out',
    'double': 'double',
    'hit_by_pitch': 'hit_by_pitch',
    'home_run': 'home_run',
    'grounded_into_double_play': 'field_out',
    'fielders_choice_out': 'field_out',
    'fielders_choice': 'field_out',
    'field_error': 'field_out',
    'double_play': 'field_out',
    'sac_fly': 'field_out',
    'triple': 'triple',
    'sac_bunt': 'field_out',
    'wild_pitch': 'wild_pitch',
    'sac_fly_double_play': 'field_out',
    'triple_play': 'field_out',
    'other_out': 'field_out',
    'sac_bunt_double_play': 'field_out',
}

In [24]:
swing_descriptions = ['foul_bunt', 'foul', 'hit_into_play', 'swinging_strike', 'foul_tip', 'swinging_strike_blocked', 'missed_bunt', 'bunt_foul_tip', 'foul_pitchout']

In [25]:
def df_clean(df):
    """
    Clean and transform baseball data using Polars.
    
    Args:
        df: Polars DataFrame with baseball pitch data
        
    Returns:
        Cleaned and transformed Polars DataFrame
    """
    
    # Create new column with grouped descriptions
    df = df.with_columns([
        pl.col("description").replace_strict(des_dict, default = None).alias("des_new")
    ])
    
    # Create new column with grouped events for hit_into_play cases
    df = df.with_columns([
        pl.when(pl.col("des_new") == "hit_into_play")
        .then(pl.col("events").replace_strict(ev_dict, default = None))
        .otherwise(None)
        .alias("ev_new")
    ])
    
    # Replace des_new with ev_new for hit_into_play cases
    df = df.with_columns([
        pl.when(pl.col("des_new") == "hit_into_play")
        .then(pl.col("ev_new"))
        .otherwise(pl.col("des_new"))
        .alias("des_new")
    ]).drop("ev_new")
    
    # Filter out null des_new values
    df = df.filter(pl.col("des_new").is_not_null())
    
    # Filter rare cases (strikes <= 2 and balls <= 3)
    df = df.filter(
        (pl.col("strikes") <= 2) & 
        (pl.col("balls") <= 3)
    )
    
    # Create count column
    df = df.with_columns([
        (pl.col("balls").cast(pl.Utf8) + "-" + pl.col("strikes").cast(pl.Utf8)).alias("count")
    ])
    
    # Convert count to categorical
    df = df.with_columns([
        pl.col("count").cast(pl.Categorical)
    ])

    # Convert p_throws to categorical
    df = df.with_columns([
        pl.col("p_throws").cast(pl.Categorical)
    ])

    # Convert stand to categorical
    df = df.with_columns([
        pl.col("stand").cast(pl.Categorical)
    ])

    # Cast the year column to integer type
    df = df.with_columns(
        pl.col('game_year').cast(pl.Int64)
    )
    
    # Calculate average run expectancy by outcome and count
    des_values = df.group_by(["des_new", "count", "p_throws", "stand"], maintain_order = True).agg([
        pl.col("delta_run_exp").mean().alias("target")
    ])
    
    # Join the average run expectancies back to the main dataset
    df = df.join(
        des_values, 
        on = ["des_new", "count"], 
        how = "left"
    )
    
    # Add binary column for swings
    df = df.with_columns([
        pl.col("description").is_in(swing_descriptions).alias("swing")
    ])
    
    return df


In [26]:
def feature_engineering(df: pl.DataFrame) -> pl.DataFrame:
    # Mirror horizontal break for left-handed pitchers
    df = df.with_columns(
        pl.when(pl.col('p_throws') == "L")
        .then(-pl.col('ax'))
        .otherwise(pl.col('ax'))
        .alias('ax')
        )
    
    # Mirror horizontal release point for left-handed pitchers
    df = df.with_columns(
        pl.when(pl.col('p_throws') == "L")
        .then(-pl.col('release_pos_x'))
        .otherwise(pl.col('release_pos_x'))
        .alias('release_pos_x')
        )
    
    # Define the pitch types to be considered
    fastball_pitch_types = ['SI', 'FF', 'FC']

    df_filtered = df.filter(pl.col('pitch_type').is_in(fastball_pitch_types))

    # Group by pitcher_id and year, then aggregate to calculate average speed and usage percentage
    df_agg = df_filtered.group_by(['pitcher', 'game_year', 'pitch_type']).agg([
        pl.col('release_speed').mean().alias('avg_fastball_speed'),
        pl.col('az').mean().alias('avg_fastball_az'),
        pl.col('ax').mean().alias('avg_fastball_ax'),
        pl.len().alias('count')
    ])

    # Sort the aggregated data by count and average fastball speed
    df_agg = df_agg.sort(['count', 'avg_fastball_speed'], descending = [True, True])
    df_agg = df_agg.unique(subset = ['pitcher', 'game_year'])

    # Join the aggregated data with the main DataFrame
    df = df.join(df_agg, on = ['pitcher', 'game_year'])

    # Find the fastest pitch type for each pitcher
    fastest_pitch_info = (
        df.group_by('pitcher')
        .agg([
            pl.col('release_speed').max().alias('max_speed')
        ])
        .join(df, on = 'pitcher')
        .filter(pl.col('release_speed') == pl.col('max_speed'))
        .group_by('pitcher')
        .agg([
            pl.col('pitch_type').first().alias('fastest_pitch_type')
        ])
    )

    # If no fastball, use the fastest pitch for avg_fastball_speed
    avg_fastest_speed = (
        df.join(fastest_pitch_info, on = 'pitcher')
        .filter(pl.col('pitch_type') == pl.col('fastest_pitch_type'))
        .group_by('pitcher')
        .agg([pl.col('release_speed').mean().alias('avg_fastest_pitch_speed')])
    )

    df = (
        df.join(avg_fastest_speed, on = 'pitcher', how = 'left')
        .with_columns([
            pl.when(pl.col('avg_fastball_speed').is_null())
            .then(pl.col('avg_fastest_pitch_speed'))
            .otherwise(pl.col('avg_fastball_speed'))
            .alias('avg_fastball_speed')
        ])
        .drop('avg_fastest_pitch_speed')
    )

    # If no fastball, use the fastest pitch for avg_fastball_az
    avg_fastest_az = (
        df.join(fastest_pitch_info, on = 'pitcher')
        .filter(pl.col('pitch_type') == pl.col('fastest_pitch_type'))
        .group_by('pitcher')
        .agg([pl.col('az').mean().alias('avg_fastest_pitch_az')])
    )

    df = (
        df.join(avg_fastest_az, on = 'pitcher', how = 'left')
        .with_columns([
            pl.when(pl.col('avg_fastball_az').is_null())
            .then(pl.col('avg_fastest_pitch_az'))
            .otherwise(pl.col('avg_fastball_az'))
            .alias('avg_fastball_az')
        ])
        .drop('avg_fastest_pitch_az')
    )

    # If no fastball, use the fastest pitch for avg_fastball_ax
    avg_fastest_ax = (
        df.join(fastest_pitch_info, on = 'pitcher')
        .filter(pl.col('pitch_type') == pl.col('fastest_pitch_type'))
        .group_by('pitcher')
        .agg([pl.col('ax').mean().alias('avg_fastest_pitch_ax')])
    )

    df = (
        df.join(avg_fastest_ax, on = 'pitcher', how = 'left')
        .with_columns([
            pl.when(pl.col('avg_fastball_ax').is_null())
            .then(pl.col('avg_fastest_pitch_ax'))
            .otherwise(pl.col('avg_fastball_ax'))
            .alias('avg_fastball_ax')
        ])
        .drop('avg_fastest_pitch_ax')
    )

    # Calculate pitch differentials
    df = df.with_columns(
        (pl.col('release_speed') - pl.col('avg_fastball_speed')).alias('speed_diff'),
        (pl.col('az') - pl.col('avg_fastball_az')).alias('az_diff'),
        (pl.col('ax') - pl.col('avg_fastball_ax')).alias('ax_diff')
    )

    return df

In [27]:
# Load data using Polars
data_2021 = pl.read_csv('./data/2021_data.csv')
data_2022 = pl.read_csv('./data/2022_data.csv')
data_2023 = pl.read_csv('./data/2023_data.csv')
data_2024 = pl.read_csv('./data/2024_data.csv')

In [28]:
all_train_data = pl.concat([data_2021, data_2022, data_2023, data_2024], how = 'vertical_relaxed')

In [29]:
all_train_data.shape

(2845162, 118)

In [30]:
# Transform the datasets
processed_all_train_data = df_clean(all_train_data).filter(pl.col("pitch_type").is_not_null())

In [ ]:
# Do feature engineering
processed_all_train_data = feature_engineering(processed_all_train_data)

In [32]:
processed_all_train_data.write_csv('./stuff_model/data/train_data.csv')